# GreenMIR Dataset Analysis
Analysis of the 113 GreenMIR articles extracted via GROBID — section structure and text cleaning pipeline.

## 1. Imports & Setup

In [1]:
import glob, re, os
import xml.etree.ElementTree as ET
from collections import Counter

## 2. Load XML Files

In [2]:
NS = {'tei': 'http://www.tei-c.org/ns/1.0'}
xml_files = sorted(glob.glob('../data/GreenMIR/text_xml/*.grobid.tei.xml'))
print(f'files found : {len(xml_files)}')

files found : 113


## 3. Extract Section Titles
Extract all `<head>` tags from the body of each article and normalize them (remove numbering, uppercase).

In [3]:
all_heads = []

for xml_file in xml_files:
    try:
        tree = ET.parse(xml_file)
        root = tree.getroot()
        body = root.find('.//tei:body', NS)
        if body is not None:
            for head in body.findall('.//tei:head', NS):
                text = (head.text or '').strip()
                if text:
                    normalized = re.sub(r'^\d+(\.\d+)*\.?\s*', '', text).strip().upper()
                    if normalized:
                        all_heads.append(normalized)
    except Exception as e:
        print(f'ERROR: {xml_file} -> {e}')

print(f'total sections: {len(all_heads)}')

total sections: 2618


## 4. Section Frequency (raw)

In [4]:
counter = Counter(all_heads)
n = len(xml_files)

print(f"{'Rank':<5} {'Count':>5} {'%':>7}  Title")
print('-' * 55)
for i, (title, count) in enumerate(counter.most_common(30), 1):
    pct = count / n * 100
    print(f"{i:<5} {count:>5} ({pct:5.1f}%)  {title}")

Rank  Count       %  Title
-------------------------------------------------------
1       108 ( 95.6%)  INTRODUCTION
2        71 ( 62.8%)  CONCLUSION
3        63 ( 55.8%)  RELATED WORK
4        50 ( 44.2%)  FIGURE 2 .
5        50 ( 44.2%)  FIGURE 1 .
6        44 ( 38.9%)  EXPERIMENTS
7        44 ( 38.9%)  FIGURE 1 :
8        42 ( 37.2%)  FIGURE 3 .
9        41 ( 36.3%)  FIGURE 2 :
10       37 ( 32.7%)  FIGURE 3 :
11       37 ( 32.7%)  TABLE 1 .
12       35 ( 31.0%)  TABLE 1 :
13       34 ( 30.1%)  FIGURE 4 .
14       33 ( 29.2%)  TABLE 2 .
15       31 ( 27.4%)  FIGURE 4 :
16       28 ( 24.8%)  EVALUATION
17       26 ( 23.0%)  RESULTS
18       26 ( 23.0%)  METHODOLOGY
19       23 ( 20.4%)  DATASET
20       21 ( 18.6%)  TABLE 2 :
21       21 ( 18.6%)  METHOD
22       20 ( 17.7%)  FIGURE 5 .
23       20 ( 17.7%)  FIGURE 5 :
24       18 ( 15.9%)  TABLE 3 .
25       18 ( 15.9%)  TRAINING
26       17 ( 15.0%)  OBJECTIVE EVALUATION
27       16 ( 14.2%)  SUBJECTIVE EVALUATION
28       16 ( 14

## 5. Filter Noise
Remove figure/table captions mistagged as section headers by GROBID, titles too short, and titles appearing only once.

In [5]:
def is_noise(title):
    if re.match(r'^(FIGURE|FIG\.|TABLE|TAB\.)\s*\d', title):
        return True
    if len(title) <= 2:
        return True
    return False

real_heads = [h for h in all_heads if not is_noise(h)]
counter_clean = Counter(real_heads)
counter_clean = Counter({title: count for title, count in counter_clean.items() if count > 1})

print(f"Before filtering : {len(all_heads)} sections")
print(f"After filtering  : {sum(counter_clean.values())} sections")
print(f"Unique titles    : {len(counter_clean)}")

Before filtering : 2618 sections
After filtering  : 880 sections
Unique titles    : 117


## 6. Section Frequency (cleaned)

In [6]:
print(f"{'Rank':<5} {'Count':>5} {'%':>7}  Title")
print('-' * 55)
for i, (title, count) in enumerate(counter_clean.most_common(), 1):
    pct = count / n * 100
    print(f"{i:<5} {count:>5} ({pct:5.1f}%)  {title}")

Rank  Count       %  Title
-------------------------------------------------------
1       108 ( 95.6%)  INTRODUCTION
2        71 ( 62.8%)  CONCLUSION
3        63 ( 55.8%)  RELATED WORK
4        44 ( 38.9%)  EXPERIMENTS
5        28 ( 24.8%)  EVALUATION
6        26 ( 23.0%)  RESULTS
7        26 ( 23.0%)  METHODOLOGY
8        23 ( 20.4%)  DATASET
9        21 ( 18.6%)  METHOD
10       18 ( 15.9%)  TRAINING
11       17 ( 15.0%)  OBJECTIVE EVALUATION
12       16 ( 14.2%)  SUBJECTIVE EVALUATION
13       16 ( 14.2%)  MODEL
14       15 ( 13.3%)  CONCLUSIONS
15       15 ( 13.3%)  EXPERIMENTAL SETUP
16       15 ( 13.3%)  CONCLUSION AND FUTURE WORK
17       14 ( 12.4%)  DATASETS
18       14 ( 12.4%)  BACKGROUND
19       11 (  9.7%)  IMPLEMENTATION DETAILS
20       11 (  9.7%)  DATA REPRESENTATION
21       11 (  9.7%)  RESULTS AND DISCUSSION
22        9 (  8.0%)  DISCUSSION
23        9 (  8.0%)  METHODS
24        9 (  8.0%)  DATA
25        9 (  8.0%)  ABLATION STUDY
26        9 (  8.0%)  MODEL ARC

## 7. Text Cleaning Pipeline
Define blacklist of sections to remove, and cleaning functions to strip citations and URLs from paragraphs.

In [7]:
BLACKLIST = [
    'related work',
    'references',
    'ablation',
    'future work',
    'appendix',
    'listening test',
    'user study',
    'subjective evaluation',
]

In [8]:
def is_blacklisted(section_name):
    s = section_name.lower()
    return any(kw in s for kw in BLACKLIST)


def clean_text(text):
    # Remove bibliographic references like [1], [2, 3], [12-15]
    text = re.sub(r'\[\d+([,\-\s]\d+)*\]', '', text)
    # Remove URLs
    text = re.sub(r'https?://\S+', '', text)
    # Remove extra whitespace
    text = re.sub(r' {2,}', ' ', text).strip()
    return text


def clean_article(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    body = root.find('.//tei:body', NS)

    if body is None:
        return ""

    # Build parent map once for the whole tree
    parent_map = {child: parent for parent in tree.iter() for child in parent}

    def get_parent(elem):
        return parent_map.get(elem)

    def should_skip_p(p):
        """Skip <p> if inside figure, table, formula, or footnote/other notes."""
        parent = get_parent(p)
        if parent is None:
            return False
        ptag = parent.tag.replace('{http://www.tei-c.org/ns/1.0}', '')
        if ptag in {'figure', 'table', 'formula'}:
            return True
        if ptag == 'note':
            # Only skip foot and other notes (page headers, figure labels)
            # Keep notes with no type — may contain useful content
            ntype = parent.get('place') or parent.get('type') or 'NO_TYPE'
            return ntype in {'foot', 'other'}
        return False

    paragraphs = []
    for div in body.findall('.//tei:div', NS):
        head = div.find('tei:head', NS)

        # Get section name
        section_name = ""
        if head is not None and head.text:
            section_name = re.sub(r'^\d+(\.\d+)*\.?\s*', '', head.text).strip()

        # Skip blacklisted sections
        if is_blacklisted(section_name):
            continue

        # Extract, clean and keep paragraphs
        for p in div.findall('.//tei:p', NS):
            if should_skip_p(p):
                continue
            text = clean_text(' '.join(p.itertext()).strip())
            if text:
                paragraphs.append(text)

    return "\n".join(paragraphs)

## 8. Apply Cleaning to All Articles
Save cleaned texts to `data/GreenMIR/text_xml_clean/`.

In [9]:
OUTPUT_DIR = '../data/GreenMIR/clean_greenmir/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

stats = {'total': 0, 'empty': 0, 'saved': 0}

for xml_file in xml_files:
    article_id = int(re.search(r'article_(\d+)', xml_file).group(1))
    stats['total'] += 1

    text = clean_article(xml_file)

    if not text.strip():
        stats['empty'] += 1
        print(f"WARNING: article_{article_id} is empty after cleaning")
        continue

    out_path = os.path.join(OUTPUT_DIR, f'article_{article_id}.txt')
    with open(out_path, 'w', encoding='utf-8') as f:
        f.write(text)
    stats['saved'] += 1

print(f"Done.")
print(f"  Total    : {stats['total']}")
print(f"  Saved    : {stats['saved']}")
print(f"  Empty    : {stats['empty']}")

Done.
  Total    : 113
  Saved    : 113
  Empty    : 0
